# 🤖 Pipeline ML Automatisé
---
Ce notebook permet de :
1. Charger n'importe quel fichier CSV
2. Sélectionner interactivement les colonnes et la cible
3. Détecter automatiquement si c'est une Classification ou une Régression
4. Entraîner et benchmarker 6 algorithmes avec cross-validation
5. Visualiser les performances avec Seaborn
6. Sauvegarder le meilleur modèle pour Streamlit

In [ ]:
# === IMPORTS ===
import pandas as pd
import numpy as np
import os
import json
import warnings
warnings.filterwarnings('ignore')

# Visualization
import seaborn as sns
import matplotlib.pyplot as plt

# Interactive Widgets
import ipywidgets as widgets
from IPython.display import display, clear_output

# Preprocessing
from sklearn.model_selection import train_test_split, cross_validate, learning_curve
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Models
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor
from sklearn.svm import SVC, SVR
from sklearn.neural_network import MLPClassifier, MLPRegressor
from xgboost import XGBClassifier, XGBRegressor
from lightgbm import LGBMClassifier, LGBMRegressor

# Metrics
from sklearn.metrics import (
    accuracy_score, recall_score, f1_score, confusion_matrix,
    mean_squared_error, mean_absolute_error, r2_score
)

# Save
import joblib

print("✅ Imports chargés avec succès !")

## 📁 Étape 1 : Chargement du CSV
Saisissez le chemin vers votre fichier CSV dans le champ ci-dessous, puis cliquez sur **Charger**.

In [ ]:
# === CHARGEMENT INTERACTIF DU CSV ===
df = None

# List available CSV files in data/ folder
data_dir = 'data'
csv_files = [f for f in os.listdir(data_dir) if f.endswith('.csv')] if os.path.exists(data_dir) else []

if csv_files:
    print(f"📂 Fichiers CSV disponibles dans '{data_dir}/' :")
    for i, f in enumerate(csv_files):
        print(f"   {i+1}. {f}")

csv_path_input = widgets.Text(
    value=f'data/{csv_files[0]}' if csv_files else 'data/mon_fichier.csv',
    placeholder='Chemin vers le fichier CSV',
    description='Fichier CSV :',
    style={'description_width': '100px'},
    layout=widgets.Layout(width='600px')
)

load_button = widgets.Button(description='📥 Charger le CSV', button_style='primary',
                             layout=widgets.Layout(width='200px'))
output_load = widgets.Output()

def on_load_click(b):
    global df
    with output_load:
        clear_output()
        path = csv_path_input.value
        if not os.path.exists(path):
            print(f"❌ Fichier introuvable : {path}")
            return
        df = pd.read_csv(path)
        print(f"✅ Fichier chargé : {path}")
        print(f"   → {df.shape[0]} lignes × {df.shape[1]} colonnes")
        print(f"\n📊 Aperçu des données :")
        display(df.head())
        print(f"\n📋 Types des colonnes :")
        display(df.dtypes)

load_button.on_click(on_load_click)
display(csv_path_input, load_button, output_load)

## 🎯 Étape 2 : Sélection des Colonnes
1. **Cochez les colonnes** que vous souhaitez utiliser comme features (variables explicatives).
2. **Choisissez la colonne cible** (Y) dans le menu déroulant.

In [ ]:
# === SELECTION INTERACTIVE DES COLONNES ===
if df is None:
    print("⚠️ Veuillez d'abord charger un fichier CSV (Étape 1).")
else:
    # Target column selector
    target_selector = widgets.Dropdown(
        options=df.columns.tolist(),
        value=df.columns[-1],
        description='🎯 Cible (Y) :',
        style={'description_width': '120px'},
        layout=widgets.Layout(width='500px')
    )

    # Feature columns selector (checkboxes)
    feature_checkboxes = {}
    checkbox_widgets = []
    for col in df.columns:
        cb = widgets.Checkbox(
            value=True,
            description=f"{col} ({df[col].dtype})",
            style={'description_width': '300px'},
            layout=widgets.Layout(width='450px')
        )
        feature_checkboxes[col] = cb
        checkbox_widgets.append(cb)

    # Update: uncheck target column automatically
    def on_target_change(change):
        for col, cb in feature_checkboxes.items():
            if col == change['new']:
                cb.value = False
            elif col == change['old'] and not cb.value:
                cb.value = True

    target_selector.observe(on_target_change, names='value')
    # Initial: uncheck the default target
    feature_checkboxes[df.columns[-1]].value = False

    print("📌 Sélectionnez la colonne cible :")
    display(target_selector)
    print("\n✅ Cochez les colonnes à GARDER comme features :")
    display(widgets.VBox(checkbox_widgets))

In [ ]:
# === VALIDATION DE LA SELECTION ===
target_col = target_selector.value
selected_features = [col for col, cb in feature_checkboxes.items() if cb.value and col != target_col]

print(f"🎯 Colonne cible : {target_col}")
print(f"📊 Features sélectionnées ({len(selected_features)}) :")
for f in selected_features:
    print(f"   • {f}")

X = df[selected_features].copy()
y = df[target_col].copy()

print(f"\n✅ X shape: {X.shape}, y shape: {y.shape}")

## 🔍 Étape 3 : Détection Automatique (Classification / Régression)

In [ ]:
# === DETECTION AUTO DU TYPE DE TACHE ===
def detect_task_type(y_series, threshold=20):
    """
    Détecte automatiquement si la tâche est une classification ou une régression.
    - Si la cible est catégorielle (object/bool) -> Classification
    - Si la cible est numérique avec peu de valeurs uniques -> Classification
    - Sinon -> Régression
    """
    if y_series.dtype == 'object' or y_series.dtype == 'bool':
        return 'classification'
    n_unique = y_series.nunique()
    if n_unique <= threshold:
        return 'classification'
    return 'regression'

task_type = detect_task_type(y)

if task_type == 'classification':
    print(f"🏷️ Tâche détectée : CLASSIFICATION")
    print(f"   → {y.nunique()} classes : {y.unique().tolist()}")
else:
    print(f"📈 Tâche détectée : RÉGRESSION")
    print(f"   → {y.nunique()} valeurs uniques (min={y.min():.2f}, max={y.max():.2f})")

## ⚙️ Étape 4 : Preprocessing Automatique

In [ ]:
# === PREPROCESSING PIPELINE ===
# Identify column types
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

print(f"🔢 Colonnes numériques ({len(numeric_cols)}) : {numeric_cols}")
print(f"🔤 Colonnes catégorielles ({len(categorical_cols)}) : {categorical_cols}")

# Build preprocessing pipelines
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_cols),
        ('cat', categorical_transformer, categorical_cols)
    ],
    remainder='drop'
)

# Encode target for classification if needed
from sklearn.preprocessing import LabelEncoder
label_encoder = None
if task_type == 'classification' and y.dtype == 'object':
    label_encoder = LabelEncoder()
    y = pd.Series(label_encoder.fit_transform(y), name=target_col)
    print(f"\n🏷️ Encodage de la cible : {dict(zip(label_encoder.classes_, label_encoder.transform(label_encoder.classes_)))}")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"\n✅ Split réalisé : Train={X_train.shape[0]} | Test={X_test.shape[0]}")

## 🏋️ Étape 5 : Entraînement & Cross-Validation (6 Algorithmes)

In [ ]:
# === DEFINITION DES MODELES ===
if task_type == 'classification':
    models = {
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
        'XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'),
        'LightGBM': LGBMClassifier(n_estimators=100, random_state=42, verbose=-1),
        'SVM': SVC(kernel='rbf', random_state=42, probability=True),
        'MLP (Neural Net)': MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42, early_stopping=True)
    }
    scoring_cv = ['accuracy', 'recall_weighted', 'f1_weighted']
    main_metric = 'f1_weighted'
else:
    models = {
        'Random Forest': RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1),
        'Gradient Boosting': GradientBoostingRegressor(n_estimators=100, random_state=42),
        'XGBoost': XGBRegressor(n_estimators=100, random_state=42),
        'LightGBM': LGBMRegressor(n_estimators=100, random_state=42, verbose=-1),
        'SVM': SVR(kernel='rbf'),
        'MLP (Neural Net)': MLPRegressor(hidden_layer_sizes=(128, 64), max_iter=500, random_state=42, early_stopping=True)
    }
    scoring_cv = ['r2', 'neg_mean_absolute_error', 'neg_root_mean_squared_error']
    main_metric = 'r2'

print(f"📋 Modèles à entraîner ({task_type.upper()}) :")
for name in models:
    print(f"   • {name}")

In [ ]:
# === ENTRAINEMENT & CROSS-VALIDATION ===
results = {}
trained_pipelines = {}
cv_results_all = {}

print("🚀 Entraînement en cours...\n")
print(f"{'='*70}")

for name, model in models.items():
    print(f"\n🔄 {name}...", end=" ")

    # Create full pipeline (preprocessing + model)
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', model)
    ])

    # Cross-validation (5-fold)
    try:
        cv_result = cross_validate(pipe, X_train, y_train, cv=5, scoring=scoring_cv,
                                   return_train_score=True, n_jobs=-1)
        cv_results_all[name] = cv_result
    except Exception as e:
        print(f"⚠️ Erreur CV : {e}")
        cv_results_all[name] = None

    # Train on full training set
    pipe.fit(X_train, y_train)
    trained_pipelines[name] = pipe

    # Predict on test set
    y_pred = pipe.predict(X_test)

    # Calculate metrics
    if task_type == 'classification':
        acc = accuracy_score(y_test, y_pred)
        rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
        f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
        cm = confusion_matrix(y_test, y_pred)
        results[name] = {'Accuracy': acc, 'Recall': rec, 'F1-Score': f1, 'Confusion Matrix': cm}
        print(f"✅ Accuracy={acc:.4f} | Recall={rec:.4f} | F1={f1:.4f}")
    else:
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        mae = mean_absolute_error(y_test, y_pred)
        r2 = r2_score(y_test, y_pred)
        results[name] = {'RMSE': rmse, 'MAE': mae, 'R²': r2}
        print(f"✅ R²={r2:.4f} | RMSE={rmse:.4f} | MAE={mae:.4f}")

print(f"\n{'='*70}")
print("\n🎉 Entraînement terminé !")

## 📊 Étape 6 : Benchmark & Visualisations Seaborn

In [ ]:
# === TABLEAU RECAPITULATIF ===
if task_type == 'classification':
    summary_data = {name: {k: v for k, v in metrics.items() if k != 'Confusion Matrix'}
                    for name, metrics in results.items()}
else:
    summary_data = results

results_df = pd.DataFrame(summary_data).T
results_df = results_df.round(4)

print("📊 Tableau récapitulatif des performances :\n")
display(results_df)

# Highlight best model
if task_type == 'classification':
    best_model_name = results_df['F1-Score'].idxmax()
    best_score = results_df.loc[best_model_name, 'F1-Score']
    print(f"\n🏆 Meilleur modèle : {best_model_name} (F1-Score = {best_score:.4f})")
else:
    best_model_name = results_df['R²'].idxmax()
    best_score = results_df.loc[best_model_name, 'R²']
    print(f"\n🏆 Meilleur modèle : {best_model_name} (R² = {best_score:.4f})")

In [ ]:
# === VISUALISATION 1 : BARPLOTS DES METRIQUES ===
sns.set_theme(style='whitegrid', palette='viridis')

if task_type == 'classification':
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    metrics_to_plot = ['Accuracy', 'Recall', 'F1-Score']
else:
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    metrics_to_plot = ['R²', 'RMSE', 'MAE']

for i, metric in enumerate(metrics_to_plot):
    ax = axes[i]
    values = results_df[metric].sort_values(ascending=(metric in ['RMSE', 'MAE']))
    colors = ['#2ecc71' if v == values.max() else '#3498db' for v in values] if metric not in ['RMSE', 'MAE'] \
        else ['#2ecc71' if v == values.min() else '#3498db' for v in values]
    sns.barplot(x=values.values, y=values.index, ax=ax, palette=colors)
    ax.set_title(f'{metric}', fontsize=14, fontweight='bold')
    ax.set_xlabel(metric)
    for j, v in enumerate(values.values):
        ax.text(v + 0.001, j, f'{v:.4f}', va='center', fontsize=10)

plt.suptitle('📊 Comparaison des Modèles', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('models/benchmark_barplots.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : models/benchmark_barplots.png")

In [ ]:
# === VISUALISATION 2 : BOXPLOTS DES SCORES DE CROSS-VALIDATION ===
cv_data = []
if task_type == 'classification':
    cv_metric_key = 'test_f1_weighted'
    cv_metric_label = 'F1-Score (CV)'
else:
    cv_metric_key = 'test_r2'
    cv_metric_label = 'R² (CV)'

for name, cv_res in cv_results_all.items():
    if cv_res is not None:
        for score in cv_res[cv_metric_key]:
            cv_data.append({'Modèle': name, cv_metric_label: score})

cv_df = pd.DataFrame(cv_data)

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxplot(data=cv_df, x='Modèle', y=cv_metric_label, ax=ax, palette='Set2')
sns.stripplot(data=cv_df, x='Modèle', y=cv_metric_label, ax=ax, color='black', alpha=0.5, size=6)
ax.set_title(f'📦 Distribution des Scores de Cross-Validation (5-Fold)', fontsize=14, fontweight='bold')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.savefig('models/cv_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : models/cv_boxplots.png")

In [ ]:
# === VISUALISATION 3 : MATRICES DE CONFUSION (CLASSIFICATION UNIQUEMENT) ===
if task_type == 'classification':
    n_models = len(results)
    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    axes = axes.flatten()

    for i, (name, metrics) in enumerate(results.items()):
        cm = metrics['Confusion Matrix']
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i],
                    xticklabels=label_encoder.classes_ if label_encoder else 'auto',
                    yticklabels=label_encoder.classes_ if label_encoder else 'auto')
        axes[i].set_title(name, fontsize=12, fontweight='bold')
        axes[i].set_xlabel('Prédit')
        axes[i].set_ylabel('Réel')

    # Hide empty subplots if less than 6 models
    for j in range(n_models, len(axes)):
        axes[j].set_visible(False)

    plt.suptitle('🔍 Matrices de Confusion', fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig('models/confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("💾 Graphique sauvegardé : models/confusion_matrices.png")
else:
    print("ℹ️ Les matrices de confusion ne sont disponibles qu'en mode Classification.")

In [ ]:
# === VISUALISATION 4 : COURBES D'APPRENTISSAGE ===
fig, axes = plt.subplots(2, 3, figsize=(20, 12))
axes = axes.flatten()

scoring_lc = 'f1_weighted' if task_type == 'classification' else 'r2'

for i, (name, pipe) in enumerate(trained_pipelines.items()):
    print(f"📈 Calcul courbe d'apprentissage : {name}...", end=" ")
    try:
        train_sizes, train_scores, val_scores = learning_curve(
            pipe, X_train, y_train, cv=5, scoring=scoring_lc,
            train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
        )

        train_mean = train_scores.mean(axis=1)
        train_std = train_scores.std(axis=1)
        val_mean = val_scores.mean(axis=1)
        val_std = val_scores.std(axis=1)

        ax = axes[i]
        ax.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.1, color='#2196F3')
        ax.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.1, color='#FF9800')
        ax.plot(train_sizes, train_mean, 'o-', color='#2196F3', label='Train')
        ax.plot(train_sizes, val_mean, 'o-', color='#FF9800', label='Validation')
        ax.set_title(name, fontsize=12, fontweight='bold')
        ax.set_xlabel("Taille de l'échantillon")
        ax.set_ylabel(scoring_lc.replace('_', ' ').title())
        ax.legend(loc='lower right')
        ax.grid(True, alpha=0.3)
        print("✅")
    except Exception as e:
        axes[i].text(0.5, 0.5, f'Erreur:\n{str(e)[:50]}', ha='center', va='center', transform=axes[i].transAxes)
        axes[i].set_title(name, fontsize=12)
        print(f"⚠️ {e}")

plt.suptitle("📈 Courbes d'Apprentissage", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('models/learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print("💾 Graphique sauvegardé : models/learning_curves.png")

## 💾 Étape 7 : Sauvegarde du Meilleur Modèle

In [ ]:
# === SAUVEGARDE DU MEILLEUR MODELE ===
best_pipeline = trained_pipelines[best_model_name]

# Save model
model_path = 'models/best_model.joblib'
joblib.dump(best_pipeline, model_path)
print(f"💾 Modèle sauvegardé : {model_path}")

# Save metadata
metadata = {
    'model_name': best_model_name,
    'task_type': task_type,
    'target_column': target_col,
    'feature_columns': selected_features,
    'numeric_columns': numeric_cols,
    'categorical_columns': categorical_cols,
    'label_classes': label_encoder.classes_.tolist() if label_encoder else None,
    'best_score': float(best_score),
    'all_results': {name: {k: float(v) if not isinstance(v, np.ndarray) else v.tolist()
                           for k, v in metrics.items()}
                    for name, metrics in results.items()}
}

metadata_path = 'models/model_metadata.json'
with open(metadata_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print(f"📄 Métadonnées sauvegardées : {metadata_path}")
print(f"\n🏆 Modèle sélectionné : {best_model_name}")
print(f"   → Prêt à être utilisé sur l'interface Streamlit (localhost:8501)")